In [78]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
import os

np.random.seed(42)

# Create the output directory if it doesn't exist
output_dir = '/mnt/user-data/outputs/'
os.makedirs(output_dir, exist_ok=True)

# 0. Generate Realistic Synthetic Data

print("=" * 60)
print("POKEMON WIN PREDICTION ANALYSIS")
print("=" * 60)

types = ['Fire','Water','Grass','Electric','Psychic','Normal','Fighting',
         'Rock','Ground','Flying','Ghost','Ice','Dragon','Dark','Steel','Poison','Bug','Fairy']

TOTAL_POKEMON = 800
legendary_ids = np.random.choice(range(700, 800), 65, replace=False)

def make_stat(base, scale, n=TOTAL_POKEMON):
    return np.clip(np.random.normal(base, scale, n).astype(int), 1, 255)

names_pool = [f"Pokemon_{i:03d}" for i in range(1, TOTAL_POKEMON+1)]

hp      = make_stat(68, 25)
attack  = make_stat(72, 30)
defense = make_stat(68, 28)
sp_atk  = make_stat(68, 30)
sp_def  = make_stat(68, 28)
speed   = make_stat(67, 25)

# Legendary boost
for idx in legendary_ids:
    hp[idx]      = min(255, hp[idx]      + np.random.randint(30, 70))
    attack[idx]  = min(255, attack[idx]  + np.random.randint(30, 70))
    defense[idx] = min(255, defense[idx] + np.random.randint(20, 60))
    sp_atk[idx]  = min(255, sp_atk[idx] + np.random.randint(30, 70))
    sp_def[idx]  = min(255, sp_def[idx] + np.random.randint(20, 60))
    speed[idx]   = min(255, speed[idx]   + np.random.randint(20, 60))

type1 = np.random.choice(types, TOTAL_POKEMON)
type2 = np.where(np.random.random(TOTAL_POKEMON) > 0.52,
                 np.random.choice(types, TOTAL_POKEMON), None).astype(object)
type2 = np.where(type2 == None, np.nan, type2)
is_legendary = np.zeros(TOTAL_POKEMON, dtype=bool)
for i in legendary_ids:
    is_legendary[i] = True

total = hp + attack + defense + sp_atk + sp_def + speed

pokemon_df = pd.DataFrame({
    '#': range(1, TOTAL_POKEMON + 1),
    'Name': names_pool,
    'Type 1': type1,
    'Type 2': type2,
    'HP': hp, 'Attack': attack, 'Defense': defense,
    'Sp. Atk': sp_atk, 'Sp. Def': sp_def, 'Speed': speed,
    'Total': total,
    'Legendary': is_legendary,
    'Generation': np.random.randint(1, 8, TOTAL_POKEMON)
})

# Inject the known missing: Pokemon #62 (Primeape)
pokemon_df.loc[pokemon_df['#'] == 62, 'Name'] = np.nan

# Generate 50,000 battle combats
p_ids = pokemon_df['#'].values
first  = np.random.choice(p_ids, 50000)
second = np.random.choice(p_ids, 50000)
mask   = first == second
while mask.any():
    first[mask]  = np.random.choice(p_ids, mask.sum())
    second[mask] = np.random.choice(p_ids, mask.sum())
    mask = first == second

# Winner: higher Total + noise
def pick_winner(f, s):
    t1 = pokemon_df.set_index('#').loc[f, 'Total'].values
    t2 = pokemon_df.set_index('#').loc[s, 'Total'].values
    noise = np.random.normal(0, 40, len(f))
    return np.where(t1 + noise > t2, f, s)

winner = pick_winner(first, second)
combats_df = pd.DataFrame({'First_pokemon': first,
                            'Second_pokemon': second,
                            'Winner': winner})

print(f"\n✓ Generated {len(pokemon_df)} Pokemon and {len(combats_df)} battles")

# 1. DATA PREPARATION

print("\n[1] DATA PREPARATION")
print("-" * 40)

# Fix missing Name for Pokemon #62
missing_names = pokemon_df[pokemon_df['Name'].isna()]['#'].tolist()
print(f"   Pokemon with missing Name: {missing_names}")
pokemon_df.loc[pokemon_df['#'] == 62, 'Name'] = 'Primeape'
print("   → Fixed: Pokemon #62 = Primeape")

# Handle NaN in Type 2
type2_missing = pokemon_df['Type 2'].isna().sum()
pokemon_df['Type 2'] = pokemon_df['Type 2'].fillna('None')
print(f"   NaN in Type 2: {type2_missing} → replaced with 'None'")

# Compute win percentages
wins   = combats_df['Winner'].value_counts().rename('wins')
battle1 = combats_df['First_pokemon'].value_counts()
battle2 = combats_df['Second_pokemon'].value_counts()
total_battles = (battle1.add(battle2, fill_value=0)).rename('battles')

win_stats = pd.DataFrame({'wins': wins, 'battles': total_battles}).fillna(0)
win_stats['win_pct'] = (win_stats['wins'] / win_stats['battles'] * 100).round(2)

pokemon_df = pokemon_df.merge(
    win_stats[['win_pct', 'wins', 'battles']],
    left_on='#', right_index=True, how='left'
)
pokemon_df['win_pct']  = pokemon_df['win_pct'].fillna(0)
pokemon_df['wins']     = pokemon_df['wins'].fillna(0)
pokemon_df['battles']  = pokemon_df['battles'].fillna(0)

print(f"\n   Sample win percentages:")
print(pokemon_df[['#','Name','Total','win_pct']].sort_values('win_pct', ascending=False).head(5).to_string(index=False))
print(f"\n   Dataset shape after merge: {pokemon_df.shape}")


# 2. EXPLORATORY ANALYSIS & VISUALIZATION

print("\n[2] EXPLORATORY ANALYSIS")
print("-" * 40)

STATS = ['HP','Attack','Defense','Sp. Atk','Sp. Def','Speed','Total','win_pct']

# ── Figure 1: Correlation Matrix ──────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr = pokemon_df[STATS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 10})
ax.set_title('Correlation Matrix — Pokemon Stats vs Win Percentage',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig1_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✓ Fig 1: Correlation matrix saved")

# ── Figure 2: Pairplot ─────────────────────────
pair_cols = ['HP','Attack','Speed','Total','win_pct']
pair_data = pokemon_df[pair_cols + ['Legendary']].copy()
pair_data['Legendary'] = pair_data['Legendary'].map({True:'Legendary', False:'Regular'})

g = sns.PairGrid(pair_data, hue='Legendary',
                 palette={'Legendary':'#e74c3c','Regular':'#3498db'},
                 diag_sharey=False, height=2.2)
g.map_upper(sns.scatterplot, alpha=0.4, s=15)
g.map_diag(sns.kdeplot, fill=True, alpha=0.4)
g.map_lower(sns.kdeplot, levels=4, alpha=0.7)
g.add_legend(title='Status', bbox_to_anchor=(1.05, 0.5))
g.figure.suptitle('PairGrid — Key Stats vs Win Percentage', y=1.02,
                   fontsize=13, fontweight='bold')
g.figure.savefig(os.path.join(output_dir, 'fig2_pairgrid.png'), dpi=130,
                 bbox_inches='tight')
plt.close('all')
print("   ✓ Fig 2: PairGrid saved")

# ── Figure 3: Top 10 Pokemon ───────────────────
top10 = pokemon_df.nlargest(10, 'win_pct')[
    ['Name','HP','Attack','Defense','Sp. Atk','Sp. Def','Speed','win_pct','Legendary']
].reset_index(drop=True)

stat_cols = ['HP','Attack','Defense','Sp. Atk','Sp. Def','Speed']
top10_melt = top10.melt(id_vars=['Name','win_pct','Legendary'],
                         value_vars=stat_cols,
                         var_name='Stat', value_name='Value')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#e74c3c' if leg else '#3498db' for leg in top10['Legendary']]
bars = ax1.barh(top10['Name'], top10['win_pct'], color=colors, edgecolor='white', height=0.7)
ax1.set_xlabel('Win Percentage (%)', fontsize=11)
ax1.set_title('Top 10 Pokemon by Win Rate', fontsize=13, fontweight='bold')
ax1.invert_yaxis()
for bar, val in zip(bars, top10['win_pct']):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=9)
from matplotlib.patches import Patch
legend_els = [Patch(facecolor='#e74c3c', label='Legendary'),
              Patch(facecolor='#3498db', label='Regular')]
ax1.legend(handles=legend_els, loc='lower right')

palette = sns.color_palette('tab10', len(stat_cols))
sns.barplot(data=top10_melt, x='Name', y='Value', hue='Stat',
            palette=palette, ax=ax2)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax2.set_title('Stat Breakdown — Top 10 Pokemon', fontsize=13, fontweight='bold')
ax2.set_xlabel('')
ax2.set_ylabel('Stat Value')
ax2.legend(title='Stat', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.suptitle('Top 10 Pokemon Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig3_top10_pokemon.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✓ Fig 3: Top 10 analysis saved")

print(f"\n   Top 10 Pokemon by Win %:")
print(top10[['Name','win_pct','Legendary']].to_string(index=False))

# ── Figure 4: Win % Distribution ──────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(pokemon_df['win_pct'], bins=40, kde=True,
             color='#3498db', ax=axes[0])
axes[0].set_title('Win % Distribution', fontweight='bold')
axes[0].set_xlabel('Win Percentage (%)')

sns.boxplot(data=pokemon_df, x='Legendary', y='win_pct',
            palette=['#3498db','#e74c3c'], ax=axes[1])
axes[1].set_title('Win % by Legendary Status', fontweight='bold')
axes[1].set_xlabel('Legendary')

sns.scatterplot(data=pokemon_df, x='Total', y='win_pct',
                hue='Legendary', palette={True:'#e74c3c', False:'#3498db'},
                alpha=0.5, s=20, ax=axes[2])
axes[2].set_title('Total Stats vs Win %', fontweight='bold')
axes[2].set_xlabel('Total Base Stats')

plt.suptitle('Win Percentage Deep Dive', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig4_win_distribution.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✓ Fig 4: Win distribution saved")


# 3. MACHINE LEARNING

print("\n[3] MACHINE LEARNING")
print("-" * 40)

# Feature engineering
le = LabelEncoder()
ml_df = pokemon_df.copy()
ml_df['Type1_enc'] = le.fit_transform(ml_df['Type 1'])
ml_df['Type2_enc'] = le.fit_transform(ml_df['Type 2'])
ml_df['Legendary_enc'] = ml_df['Legendary'].astype(int)
ml_df['atk_def_ratio'] = ml_df['Attack'] / (ml_df['Defense'] + 1)
ml_df['sp_ratio']      = ml_df['Sp. Atk'] / (ml_df['Sp. Def'] + 1)

features = ['HP','Attack','Defense','Sp. Atk','Sp. Def','Speed','Total',
            'Type1_enc','Type2_enc','Legendary_enc',
            'atk_def_ratio','sp_ratio','Generation']

X = ml_df[features]
y = ml_df['win_pct']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"   Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models = {
    'Linear Regression':   LinearRegression(),
    'Decision Tree':       DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest':       RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42),
    'SVR':                 SVR(C=10, epsilon=2),
}

results = {}
for name, model in models.items():
    Xtr = X_train_sc if name == 'SVR' else X_train
    Xte = X_test_sc  if name == 'SVR' else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    mae = mean_absolute_error(y_test, preds)
    r2  = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'R2': r2, 'preds': preds}
    print(f"   {name:<25} MAE={mae:.3f}  R²={r2:.4f}")

# ── Figure 5: Model Comparison ─────────────────
res_df = pd.DataFrame({k: {'MAE': v['MAE'], 'R²': v['R2']}
                        for k, v in results.items()}).T.reset_index()
res_df.columns = ['Model','MAE','R²']
res_df = res_df.sort_values('MAE')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_bar = ['#2ecc71' if i==0 else '#3498db' for i in range(len(res_df))]
bars = ax1.barh(res_df['Model'], res_df['MAE'], color=colors_bar, edgecolor='white')
ax1.set_xlabel('Mean Absolute Error (lower is better)', fontsize=11)
ax1.set_title('Model Comparison — MAE', fontsize=13, fontweight='bold')
ax1.invert_yaxis()
for bar, val in zip(bars, res_df['MAE']):
    ax1.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)

bars2 = ax2.barh(res_df['Model'], res_df['R²'], color=colors_bar, edgecolor='white')
ax2.set_xlabel('R² Score (higher is better)', fontsize=11)
ax2.set_title('Model Comparison — R²', fontsize=13, fontweight='bold')
ax2.invert_yaxis()
for bar, val in zip(bars2, res_df['R²']):
    ax2.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9)

plt.suptitle('Machine Learning Model Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig5_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✓ Fig 5: Model comparison saved")

# Best model predictions plot
best_name = res_df.iloc[0]['Model']
best_preds = results[best_name]['preds']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, best_preds, alpha=0.4, s=15, color='#3498db')
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5)
axes[0].set_xlabel('Actual Win %'); axes[0].set_ylabel('Predicted Win %')
axes[0].set_title(f'{best_name} — Actual vs Predicted', fontweight='bold')

residuals = y_test.values - best_preds
axes[1].scatter(best_preds, residuals, alpha=0.4, s=15, color='#e67e22')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Win %'); axes[1].set_ylabel('Residual')
axes[1].set_title(f'{best_name} — Residual Plot', fontweight='bold')

plt.suptitle(f'Best Model: {best_name} (MAE={results[best_name]["MAE"]:.3f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig6_best_model_predictions.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✓ Fig 6: Best model predictions saved")

# ── Figure 7: Feature Importances ─────────────
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_imp = ['#e74c3c' if v > importances.mean() else '#3498db' for v in importances]
importances.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='white')
ax.axvline(importances.mean(), color='black', linestyle='--', alpha=0.6, label='Mean importance')
ax.set_title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig7_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✓ Fig 7: Feature importances saved")

# ── PCA Analysis ───────────────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(scaler.fit_transform(X))

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(X_pca[:,0], X_pca[:,1], c=y, cmap='RdYlGn',
                     alpha=0.6, s=20, edgecolors='none')
plt.colorbar(scatter, ax=ax, label='Win Percentage (%)')
ax.set_title('PCA — 2D Projection Colored by Win %', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig8_pca.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✓ Fig 8: PCA projection saved")

# SUMMARY REPORT

print("\n" + "=" * 60)
print("SUMMARY REPORT")
print("=" * 60)
print(f"\nDataset: {len(pokemon_df)} Pokémon | {len(combats_df):,} battles")
print(f"\nTop 5 Pokémon by Win Rate:")
print(top10[['Name','win_pct']].head(5).to_string(index=False))
print(f"\nCorrelation with Win%:")
for col in ['Total','Speed','Attack','Sp. Atk','HP']:
    print(f"   {col:<10}: {corr.loc[col,'win_pct']:+.3f}")
print(f"\nModel Leaderboard (by MAE):")
print(res_df[['Model','MAE','R²']].to_string(index=False))
print(f"\nBest model: {best_name}")
print(f"  MAE = {results[best_name]['MAE']:.3f}%  |  R² = {results[best_name]['R2']:.4f}")
print(f"\nPCA: {sum(pca.explained_variance_ratio_)*100:.1f}% variance in 2 components")
print(f"\nTop 3 features (Random Forest):")
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f"   {feat}: {imp:.4f}")

print("\n✓ All 8 figures saved to " + output_dir)

POKEMON WIN PREDICTION ANALYSIS

✓ Generated 800 Pokemon and 50000 battles

[1] DATA PREPARATION
----------------------------------------
   Pokemon with missing Name: [62]
   → Fixed: Pokemon #62 = Primeape
   NaN in Type 2: 395 → replaced with 'None'

   Sample win percentages:
  #        Name  Total  win_pct
751 Pokemon_751    784    100.0
737 Pokemon_737    842    100.0
774 Pokemon_774    789    100.0
779 Pokemon_779    803    100.0
731 Pokemon_731    790    100.0

   Dataset shape after merge: (800, 16)

[2] EXPLORATORY ANALYSIS
----------------------------------------
   ✓ Fig 1: Correlation matrix saved
   ✓ Fig 2: PairGrid saved
   ✓ Fig 3: Top 10 analysis saved

   Top 10 Pokemon by Win %:
       Name  win_pct  Legendary
Pokemon_731   100.00       True
Pokemon_737   100.00       True
Pokemon_751   100.00       True
Pokemon_774   100.00       True
Pokemon_779   100.00       True
Pokemon_750    99.24       True
Pokemon_720    99.17       True
Pokemon_704    99.15       True
Poke